# Stage 2 Notebook 72 - Exp2QQQ KD from NB62 teacher + K=4 topk_fixed student

**Architectural / training change #3: knowledge distillation.** NB62 (decoded_f1=0.073, matched_iou=0.550) has the best ANCHOR-HEAD geometry on the project. NB68 (K=4 topk_fixed, decoded_f1=0.067, val_lane_best_f1=0.216) has the best CLS discrimination. They are at the two ends of the same trade-off.

Exp2QQQ: train a student model with NB68's cls recipe (K=4 topk_fixed, cls_sep, VFL), but AUGMENT the loss with distillation from NB62's geometry-strong teacher. The teacher's cls and coord_pred are passed through as soft targets via the existing `w_distill > 0` KD path in FusionLaneLoss. The student should:
1. Get NB68's cls discrimination from its own VFL + K=4 supervision
2. Get NB62's geometric precision from MSE-aligning its coord_pred to teacher's

Diffs vs NB62 (exp57):
- `teacher.lane_head_checkpoint: null -> NB62's best.pt`
- `loss.lane.w_distill: 0.0 -> 1.0`
- `lane_assigner: dynamic_k -> topk_fixed`, `topk_fixed_per_gt: 4` (NB68 cls winner)

Reference: Hinton et al. 'Distilling the Knowledge in a Neural Network' (2015); CLRKDNet uses self-distillation between its own intermediate stages as the namesake mechanism.

### Run mode
1. Smoke (verify teacher checkpoint loads).
2. 12 epochs full 70K. ~3.5-4 hr.

NOTE: The teacher checkpoint `exp57_..._best.pt` must exist in Drive. Run NB62 first if not.

In [4]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [5]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp67_rmt_gca_anchor_cls_sep_vfl_kd_from_nb62_full_data_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp67_rmt_gca_anchor_cls_sep_vfl_kd_from_nb62_full_data_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp67_rmt_gca_anchor_cls_sep_vfl_kd_from_nb62_full_data_joint_smoke.log
OK exp67_rmt_gca_anchor_cls_sep_vfl_kd_from_nb62_full_data_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=5.3438 det_loss=3.9933 grad_cos=-0.2707 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.4973390996456146, 'gate/lane_mean': 0.498223215341568, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [6]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp67_rmt_gca_anchor_cls_sep_vfl_kd_from_nb62_full_data_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full12'
    EPOCHS = 12
    BATCH_SIZE = 8
    LIMIT_TRAIN = None
    LIMIT_VAL = 2000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: None
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp67_rmt_gca_anchor_cls_sep_vfl_kd_from_nb62_full_data_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp67_rmt_gca_anchor_cls_sep_vfl_kd_from_nb62_full_data_joint_full12 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp67_rmt_gca_anchor_cls_sep_vfl_kd_from_nb62_full_data_joint_full12.tar --epochs 12 --batch-size 8 --limit-val 2000 --force-extract --print-every 50
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp67_rmt_gca_anchor_cls_sep_vfl_kd_from_nb62_full_data_joint_full12.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp67_rmt_gca_anchor_cls_sep_vfl_kd_from_nb62_full_data_joint_full12_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --c

CalledProcessError: Command '['/usr/bin/python3', '-u', 'stage2/scripts/train_joint_model_experiment.py', '--config', 'stage2/configs/exp67_rmt_gca_anchor_cls_sep_vfl_kd_from_nb62_full_data_joint.yaml', '--curve-tar', '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar', '--curve-root', '/content/bdd100k_clrkd_curve', '--work-dir', '/content/exp67_rmt_gca_anchor_cls_sep_vfl_kd_from_nb62_full_data_joint_full12', '--output-tar', '/content/drive/MyDrive/EcoCAR/training_runs/exp67_rmt_gca_anchor_cls_sep_vfl_kd_from_nb62_full_data_joint_full12.tar', '--epochs', '12', '--batch-size', '8', '--limit-val', '2000', '--force-extract', '--print-every', '50']' returned non-zero exit status 1.

## What to watch in Exp2QQQ

Reference NB62 (teacher): matched_iou=0.550, decoded_f1=0.073, val_lane_f1=0.118.
Reference NB68 (student's matcher): matched_iou=0.368, val_lane_f1=0.193, decoded_f1=0.067.

Pass criteria at epoch 12:
- val/lane/distill (the KD loss term) should DECREASE -- evidence the student is mimicking the teacher.
- val/matched_line_iou >= 0.50 (KD pulls geometry up toward teacher's 0.55).
- val/lane_f1 >= 0.15 (K=4 matcher gives discrimination, NOT below NB62's 0.118).
- val/lane/decoded_f1 >= 0.10 (40% over NB62 if KD genuinely combines the two strengths).
- pos-neg gap >= 0.05.

If decoded_f1 >= 0.10: KD from the geometry-strong teacher into a cls-strong student does combine the two trade-off endpoints. This is the architectural/training fusion that pure config-tuning could not achieve.